<a href="https://colab.research.google.com/github/Mriano29/hotel_demand_forecasting_system/blob/main/data/istac_prep/tourist_occupation/Tourist_occupation_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # **`Notebook 2:` Preparación de datos externos (Total ocupacion de turistas)**

Se utiliza un dataset procedente del Instituto Canario de Estadística (ISTAC), que proporciona información agregada sobre la actividad turística en Canarias.

Este conjunto de datos permite incorporar variables relacionadas con la demanda turística, ocupación y estacionalidad real del destino, aportando un contexto macroeconómico de gran valor para el análisis de reservas.

Antes de su integración, se realiza un proceso de exploración y limpieza para comprender su estructura, calidad y nivel de agregación.

###**Indice**

1. Carga de datos y visualización de primeras columnas
2. Inspección inicial de la estructura
3. Identificación de la cabecera real
4. Extracción de columnas temporales
5. Análisis de estructura tras reconstrucción
6. Propagación de indicadores
7. Filtrado a Canarias
8. Validación final de estructura
9. Transformación a formato longitudinal (MELT)
10. Exportado de datos

### 1 — Carga de datos y visualización de primeras columnas

Se carga el dataset de actividad hotelera procedente de ISTAC.

Este dataset no está en un formato tabular estándar, sino que contiene:
- Metadatos en las primeras filas
- Estructura tipo informe (no analítica)
- Columnas no etiquetadas correctamente

Por tanto, será necesario un proceso de reconstrucción antes de su uso.

In [20]:
import pandas as pd

url = "https://raw.githubusercontent.com/Mriano29/hotel_demand_forecasting_system/refs/heads/main/data/raw_data/total_tourist_accomodations.csv"

df_tourist_accomodations = pd.read_csv(url)

print(df_tourist_accomodations.shape)
df_tourist_accomodations.head()

(640, 226)


,"Establecimientos abiertos, plazas y habitaciones ofertadas, tasas de ocupación, tarifa media, ingresos y empleos. Islas y municipios de Canarias por periodos",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 216,Unnamed: 217,Unnamed: 218,Unnamed: 219,Unnamed: 220,Unnamed: 221,Unnamed: 222,Unnamed: 223,Unnamed: 224,Unnamed: 225
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Last update,"Apr 22, 2026",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Units:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Measure unit (Tarifa media diaria): Euro (Units),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Measure unit (Empleos): Number (Units),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2 — Inspección inicial de la estructura

Se inspeccionan las primeras filas del dataset para identificar:

- Presencia de metadatos
- Ubicación de la cabecera real
- Posible inicio de los datos

In [21]:
for i in range(25):
    print(f"\n--- FILA {i} ---")
    print(df_tourist_accomodations.iloc[i].values[:10])


--- FILA 0 ---
[nan nan nan nan nan nan nan nan nan nan]

--- FILA 1 ---
['Last update' 'Apr 22, 2026' nan nan nan nan nan nan nan nan]

--- FILA 2 ---
['Units:' nan nan nan nan nan nan nan nan nan]

--- FILA 3 ---
[nan 'Measure unit (Tarifa media diaria): Euro (Units)' nan nan nan nan
 nan nan nan nan]

--- FILA 4 ---
[nan 'Measure unit (Empleos): Number (Units)' nan nan nan nan nan nan nan
 nan]

--- FILA 5 ---
[nan
 'Measure unit (Empleos por cada 100 viajeros entrados): Percentage (Units)'
 nan nan nan nan nan nan nan nan]

--- FILA 6 ---
[nan
 'Measure unit (Empleos por cada 10.000 euros de ingresos): Por cada 10.000 (Unidades)'
 nan nan nan nan nan nan nan nan]

--- FILA 7 ---
[nan
 'Measure unit (Empleos por cada 1.000 pernoctaciones): Per each 1.000 (Units)'
 nan nan nan nan nan nan nan nan]

--- FILA 8 ---
[nan
 'Measure unit (Empleos por cada 100 plazas alojativas): Percentage (Units)'
 nan nan nan nan nan nan nan nan]

--- FILA 9 ---
[nan 'Measure unit (Establecimientos): E

###3 — Identificación de la cabecera real

Tras la inspección, se identifica que:

- La fila 20 contiene los periodos temporales (fechas)
- Las filas anteriores contienen metadatos (unidades, descripción, etc.)

Por tanto, la fila 20 se utilizará como referencia para reconstruir las columnas del dataset.

In [22]:
header_row = 20
df_tourist_accomodations.iloc[header_row].values[:10]

array([nan, nan, '03/2026', '02/2026', '01/2026', '2025', '12/2025',
       '11/2025', '10/2025', '09/2025'], dtype=object)

###4 — Extracción de columnas temporales

Se extraen las columnas correspondientes a los periodos temporales a partir de la fila identificada como cabecera.

Estas columnas representan observaciones mensuales de los distintos indicadores.

In [23]:
# Extraer fechas (columnas temporales)
date_cols = df_tourist_accomodations.iloc[header_row, 2:].values

# Crear dataset a partir de los datos reales
df_clean = df_tourist_accomodations.iloc[header_row + 1:].copy()

# Asignar nombres de columnas
df_clean.columns = ['indicator', 'location'] + list(date_cols)

df_clean = df_clean.reset_index(drop=True)

df_clean.head()

,indicator,location,03/2026,02/2026,01/2026,2025,12/2025,11/2025,10/2025,09/2025,...,10/2009,09/2009,08/2009,07/2009,06/2009,05/2009,04/2009,03/2009,02/2009,01/2009
0,Tarifa media diaria,Canary Islands,"143,74","143,46","140,3","127,26","146,81","135,51","126,59","115,77",...,"55,06","52,35","57,9","53,16","50,93","51,25","59,89","57,17","56,48","56,75"
1,NaN,Lanzarote,"141,07","136,2","129,12","126,43","138,04","124,01","125,47","119,4",...,"48,88","47,09","47,83","47,19","41,88","44,71","50,27","48,09","49,73","49,75"
2,NaN,Arrecife,"110,94","112,65","115,99","108,39","115,56","100,94","99,37","114,64",...,"62,89","44,15",42,"40,93","32,98","44,21","43,12","34,9","56,07","62,41"
3,NaN,Haría,"63,65","58,81","61,9","67,46","57,76","62,12","62,44","59,06",...,"30,61","32,65","32,5","33,26","28,66","33,24","33,63","32,18","33,25","27,92"
4,NaN,Teguise,"125,17","107,53","104,77","102,03","112,83","103,6","100,45","93,88",...,"45,69","50,72","52,74","50,61","39,47","40,86","47,65","47,01","51,32","52,7"


###5 — Análisis de estructura tras reconstrucción

Se observa que:
- La columna `indicator` contiene valores nulos en muchas filas
- Los indicadores están organizados en bloques jerárquicos
- Cada bloque corresponde a un tipo de métrica (ADR, ocupación, etc.)

In [24]:
df_clean[['indicator', 'location']].head(20)

,indicator,location
0,Tarifa media diaria,Canary Islands
1,NaN,Lanzarote
2,NaN,Arrecife
3,NaN,Haría
4,NaN,Teguise
5,NaN,Tías
6,NaN,Yaiza
7,NaN,Fuerteventura
8,NaN,Antigua
9,NaN,La Oliva


###6 — Propagación de indicadores

Se aplica forward fill para propagar los valores de la columna `indicator`.

Esto permite asignar correctamente cada fila a su métrica correspondiente.

In [25]:
df_clean['indicator'] = df_clean['indicator'].ffill()

df_clean[['indicator', 'location']].head(20)

,indicator,location
0,Tarifa media diaria,Canary Islands
1,Tarifa media diaria,Lanzarote
2,Tarifa media diaria,Arrecife
3,Tarifa media diaria,Haría
4,Tarifa media diaria,Teguise
5,Tarifa media diaria,Tías
6,Tarifa media diaria,Yaiza
7,Tarifa media diaria,Fuerteventura
8,Tarifa media diaria,Antigua
9,Tarifa media diaria,La Oliva


###7 — Filtrado a Canarias

Se filtra el dataset para mantener únicamente el nivel agregado de Canarias.

Esta decisión se toma para asegurar coherencia con el dataset principal, que no contiene información geográfica.

In [26]:
df_canarias = df_clean[df_clean['location'] == 'Canary Islands'].copy()

print(df_canarias.shape)
df_canarias.head()

(13, 226)


,indicator,location,03/2026,02/2026,01/2026,2025,12/2025,11/2025,10/2025,09/2025,...,10/2009,09/2009,08/2009,07/2009,06/2009,05/2009,04/2009,03/2009,02/2009,01/2009
0,Tarifa media diaria,Canary Islands,"143,74","143,46","140,3","127,26","146,81","135,51","126,59","115,77",...,"55,06","52,35","57,9","53,16","50,93","51,25","59,89","57,17","56,48","56,75"
47,Empleos,Canary Islands,75626,75364,75575,72120,76265,73965,74070,72974,...,51000,51082,51542,50769,50115,48113,52597,53714,54478,55118
94,Empleos por cada 100 viajeros entrados,Canary Islands,"6,08","6,51","6,67","6,06","6,64","6,32","5,83","6,46",...,"5,62","6,44","4,72","5,17","6,58","6,59","6,1","6,03","6,72","6,55"
141,Empleos por cada 10.000 euros de ingresos,Canary Islands,"1,26","1,34","1,29","1,46","1,31","1,33","1,45","1,62",...,3,"3,21","2,48","3,02","3,74","3,6","2,96","2,57","3,02","2,47"
188,Empleos por cada 1.000 pernoctaciones,Canary Islands,"8,9","9,3","8,99","8,74","9,38","9,09","8,54","9,17",...,"7,48","7,9","5,85","6,55","8,64","8,97","8,03","6,98","7,86",7


###8 — Validación final de estructura

Se valida que el dataset contiene:

- Un único nivel geográfico (Canarias)
- Indicadores correctamente propagados
- Variables temporales en formato ancho

A partir de este punto, el dataset está listo para transformación a formato longitudinal.

In [27]:
df_canarias['indicator'].value_counts()

,count
indicator,
Tarifa media diaria,1
Empleos,1
Empleos por cada 100 viajeros entrados,1
Empleos por cada 10.000 euros de ingresos,1
Empleos por cada 1.000 pernoctaciones,1
Empleos por cada 100 plazas alojativas,1
Establecimientos,1
Habitaciones,1
Ingresos totales,1


###9 — Transformación a formato longitudinal (MELT)

El dataset se encuentra actualmente en formato ancho (wide format), donde cada columna representa un periodo temporal.

Para su uso en modelos de machine learning y posterior integración con el dataset principal, es necesario transformarlo a formato longitudinal (long format).

Este formato permite:

- Representar cada observación como una combinación de indicador y fecha
- Facilitar el análisis temporal
- Simplificar la integración con otros datasets

Se utiliza la operación `melt` para realizar esta transformación.

####9.1 — Identificación de columnas temporales

Se identifican las columnas que representan periodos temporales.

Estas columnas serán utilizadas como variables de medida en la transformación.

In [28]:
# Separar columnas no temporales
id_vars = ['indicator', 'location']

# Columnas temporales (todas las demás)
value_vars = [col for col in df_canarias.columns if col not in id_vars]

len(value_vars), value_vars[:5]

(224, ['03/2026', '02/2026', '01/2026', '2025', '12/2025'])

####9.2 — Aplicación de MELT

Se aplica la transformación de formato ancho a formato largo.

Cada fila representará un valor de un indicador en un momento temporal específico.

In [29]:
df_ta_long = df_canarias.melt(
    id_vars=id_vars,
    value_vars=value_vars,
    var_name='date',
    value_name='value'
)

df_ta_long.head()

,indicator,location,date,value
0,Tarifa media diaria,Canary Islands,03/2026,"143,74"
1,Empleos,Canary Islands,03/2026,75626
2,Empleos por cada 100 viajeros entrados,Canary Islands,03/2026,"6,08"
3,Empleos por cada 10.000 euros de ingresos,Canary Islands,03/2026,"1,26"
4,Empleos por cada 1.000 pernoctaciones,Canary Islands,03/2026,"8,9"


####9.3 — Conversión de tipos de datos

Se convierten las variables a tipos adecuados:

- `date` → formato temporal
- `value` → numérico (float)

Esto es necesario para el análisis y modelado posterior.

In [30]:
# Filtrar solo fechas con formato mensual (contienen "/")
df_ta_long = df_ta_long[df_ta_long['date'].astype(str).str.contains('/')].copy()

# Convertir fechas de forma segura
df_ta_long['date'] = pd.to_datetime(df_ta_long['date'], format='%m/%Y', errors='coerce')

# Convertir valores numéricos (coma → punto)
df_ta_long['value'] = (
    df_ta_long['value']
    .astype(str)
    .str.replace(',', '.', regex=False)
)

df_ta_long['value'] = pd.to_numeric(df_ta_long['value'], errors='coerce')

df_ta_long.dtypes

,0
indicator,object
location,object
date,datetime64[ns]
value,float64


####9.4 — Validación de datos nulos

Se revisan valores nulos tras la transformación.

Los nulos pueden deberse a:

- Periodos sin datos disponibles
- Indicadores no aplicables en ciertos periodos
- Errores de conversión

In [31]:
df_ta_long.isnull().sum().sort_values(ascending=False)

,0
indicator,0
location,0
date,0
value,0


####9.5 — Validación estructural final

Se valida que el dataset cumple con:

- Estructura longitudinal correcta
- Tipos de datos adecuados
- Preparación para análisis temporal y modelado

Este dataset será utilizado posteriormente para la agregación de variables y su integración con el dataset principal de cancelaciones.

In [32]:
df_ta_long.head()

,indicator,location,date,value
0,Tarifa media diaria,Canary Islands,2026-03-01,143.74
1,Empleos,Canary Islands,2026-03-01,75626.00
2,Empleos por cada 100 viajeros entrados,Canary Islands,2026-03-01,6.08
3,Empleos por cada 10.000 euros de ingresos,Canary Islands,2026-03-01,1.26
4,Empleos por cada 1.000 pernoctaciones,Canary Islands,2026-03-01,8.90


In [33]:
df_ta_long.info()

df_ta_long['indicator'].value_counts()

<class 'pandas.core.frame.DataFrame'>
Index: 2691 entries, 0 to 2911
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   indicator  2691 non-null   object        
 1   location   2691 non-null   object        
 2   date       2691 non-null   datetime64[ns]
 3   value      2691 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 105.1+ KB


,count
indicator,
Tarifa media diaria,207
Empleos,207
Empleos por cada 100 viajeros entrados,207
Empleos por cada 10.000 euros de ingresos,207
Empleos por cada 1.000 pernoctaciones,207
Empleos por cada 100 plazas alojativas,207
Establecimientos,207
Habitaciones,207
Ingresos totales,207


####9.6 — Transformación a formato ancho


Tras la transformación a formato longitudinal, se procede a convertir el dataset a formato ancho (wide format) mediante una operación de pivot.

El objetivo es:

- Representar cada indicador como una variable (feature)
- Obtener una fila por periodo temporal
- Preparar el dataset para su uso en modelos de machine learning

Este paso es crítico para transformar los datos en una estructura adecuada para modelado.

In [34]:
df_ta_wide = df_ta_long.pivot_table(
    index='date',
    columns='indicator',
    values='value'
)

df_ta_wide.head()

indicator,Empleos,Empleos por cada 1.000 pernoctaciones,Empleos por cada 10.000 euros de ingresos,Empleos por cada 100 plazas alojativas,Empleos por cada 100 viajeros entrados,Establecimientos,Habitaciones,Ingresos por habitación,Ingresos totales,Plazas,Tarifa media diaria,Tasa de ocupación por habitación,Tasa de ocupación por plaza
date,,,,,,,,,,,,,
2009-01-01,55118.0,7.00,2.47,12.60,6.55,1872.0,177298.0,40.57,222801853.4,437453.0,56.75,71.50,58.04
2009-02-01,54478.0,7.86,3.02,12.52,6.72,1862.0,176299.0,36.51,180092848.9,435138.0,56.48,64.67,56.89
2009-03-01,53714.0,6.98,2.57,12.30,6.03,1862.0,176600.0,38.16,208754329.9,436734.0,57.17,66.76,56.86
2009-04-01,52597.0,8.03,2.96,12.10,6.10,1846.0,175818.0,33.75,177880065.3,434769.0,59.89,56.37,50.24
2009-05-01,48113.0,8.97,3.60,11.32,6.59,1798.0,172152.0,25.04,133522737.5,424943.0,51.25,48.87,40.71


####9.7 — Limpieza de nombres de columnas

Se normalizan los nombres de columnas eliminando:

- espacios
- mayúsculas
- acentos

Esto mejora la compatibilidad del dataset con herramientas de machine learning y pipelines de producción.

In [35]:
import unicodedata

def clean_column(col):
    col = col.strip().lower().replace(' ', '_')
    col = ''.join(
        c for c in unicodedata.normalize('NFD', col)
        if unicodedata.category(c) != 'Mn'
    )
    return col

df_ta_wide.columns = [clean_column(col) for col in df_ta_wide.columns]

df_ta_wide.columns = df_ta_wide.columns.str.replace('.', '_', regex=False)

df_ta_wide.head()

,empleos,empleos_por_cada_1_000_pernoctaciones,empleos_por_cada_10_000_euros_de_ingresos,empleos_por_cada_100_plazas_alojativas,empleos_por_cada_100_viajeros_entrados,establecimientos,habitaciones,ingresos_por_habitacion,ingresos_totales,plazas,tarifa_media_diaria,tasa_de_ocupacion_por_habitacion,tasa_de_ocupacion_por_plaza
date,,,,,,,,,,,,,
2009-01-01,55118.0,7.00,2.47,12.60,6.55,1872.0,177298.0,40.57,222801853.4,437453.0,56.75,71.50,58.04
2009-02-01,54478.0,7.86,3.02,12.52,6.72,1862.0,176299.0,36.51,180092848.9,435138.0,56.48,64.67,56.89
2009-03-01,53714.0,6.98,2.57,12.30,6.03,1862.0,176600.0,38.16,208754329.9,436734.0,57.17,66.76,56.86
2009-04-01,52597.0,8.03,2.96,12.10,6.10,1846.0,175818.0,33.75,177880065.3,434769.0,59.89,56.37,50.24
2009-05-01,48113.0,8.97,3.60,11.32,6.59,1798.0,172152.0,25.04,133522737.5,424943.0,51.25,48.87,40.71


####9.8 — Validación final

Se comprueba:

- Número de filas (periodos temporales)
- Número de variables (indicadores)
- Tipos de datos

In [36]:
df_ta_wide.shape
df_ta_wide.dtypes

,0
empleos,float64
empleos_por_cada_1_000_pernoctaciones,float64
empleos_por_cada_10_000_euros_de_ingresos,float64
empleos_por_cada_100_plazas_alojativas,float64
empleos_por_cada_100_viajeros_entrados,float64
establecimientos,float64
habitaciones,float64
ingresos_por_habitacion,float64
ingresos_totales,float64
plazas,float64


### 10 — Unificación con `Reservas de hotel`

####10.1 — Creación de variable temporal mensual en reservas

Se crea una variable `date` a partir de `arrival_date`, convirtiéndola al inicio de cada mes mediante la transformación a periodo mensual (`to_period("M")`) y su posterior conversión a timestamp.

Este paso permite homogeneizar la granularidad temporal del dataset de reservas con el dataset de indicadores turísticos, que está agregado a nivel mensual.


In [37]:
url = "https://raw.githubusercontent.com/Mriano29/hotel_demand_forecasting_system/refs/heads/main/data/hotel_bookings_prep/hotel_bookings_prepared.csv"

df_hb = pd.read_csv(url)

df_hb.head()

,hotel,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,...,arrival_date,total_guests,has_children,total_nights,is_long_stay,arrival_year,arrival_month,arrival_dayofweek,season,nights_per_guest
0,Resort Hotel,0,7,0,1,1,0.0,0,BB,GBR,...,2015-07-01,1.0,0,1,0,2015,7,2,Summer,1.0
1,City Hotel,0,257,0,2,1,0.0,0,HB,PRT,...,2015-07-01,1.0,0,2,0,2015,7,2,Summer,2.0
2,City Hotel,0,257,0,2,2,0.0,0,HB,PRT,...,2015-07-01,2.0,0,2,0,2015,7,2,Summer,1.0
3,City Hotel,0,257,0,2,2,0.0,0,HB,PRT,...,2015-07-01,2.0,0,2,0,2015,7,2,Summer,1.0
4,City Hotel,0,257,0,2,2,0.0,0,HB,PRT,...,2015-07-01,2.0,0,2,0,2015,7,2,Summer,1.0


In [39]:
df_hb['arrival_date'] = pd.to_datetime(df_hb['arrival_date'])
df_hb["date"] = df_hb["arrival_date"].dt.to_period("M").dt.to_timestamp()

####10.2 — Preparación del dataset de ocupación turistica

Se reinicia el índice del dataset de indicadores (`df_ta_wide`) para convertir la columna de fechas en una variable accesible como columna estándar.

Posteriormente, se asegura que la variable `date` tenga formato datetime, garantizando la compatibilidad con el dataset de reservas durante el proceso de unión.


In [40]:
df_ta_wide = df_ta_wide.reset_index()
df_ta_wide["date"] = pd.to_datetime(df_ta_wide["date"])

####10.3 — Preparación del lag en dataset del ISTAC

Se aplica un desplazamiento temporal de un mes hacia adelante en la variable `date` del dataset de indicadores turísticos.

Este ajuste (lag) permite que cada observación de reservas se asocie con información del mes anterior, evitando el uso de datos del mismo periodo en el que ocurre la reserva.


In [41]:
df_ta_wide["date"] = df_ta_wide["date"] + pd.offsets.MonthBegin(1)

####10.4 — Merge de los datasets

Se realiza la unión entre el dataset de reservas y el dataset de indicadores turísticos utilizando la variable común `date`.

El tipo de unión utilizado es un **left join**, lo que garantiza que todas las reservas se conserven y únicamente se añada la información de indicadores cuando exista correspondencia temporal.


In [42]:
df_hb_to = df_hb.merge(
    df_ta_wide,
    on="date",
    how="left"
)

df_hb_to.head()

,hotel,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,...,empleos_por_cada_100_plazas_alojativas,empleos_por_cada_100_viajeros_entrados,establecimientos,habitaciones,ingresos_por_habitacion,ingresos_totales,plazas,tarifa_media_diaria,tasa_de_ocupacion_por_habitacion,tasa_de_ocupacion_por_plaza
0,Resort Hotel,0,7,0,1,1,0.0,0,BB,GBR,...,13.37,5.81,1746.0,167738.0,42.49,213641489.9,412368.0,62.74,67.73,58.9
1,City Hotel,0,257,0,2,1,0.0,0,HB,PRT,...,13.37,5.81,1746.0,167738.0,42.49,213641489.9,412368.0,62.74,67.73,58.9
2,City Hotel,0,257,0,2,2,0.0,0,HB,PRT,...,13.37,5.81,1746.0,167738.0,42.49,213641489.9,412368.0,62.74,67.73,58.9
3,City Hotel,0,257,0,2,2,0.0,0,HB,PRT,...,13.37,5.81,1746.0,167738.0,42.49,213641489.9,412368.0,62.74,67.73,58.9
4,City Hotel,0,257,0,2,2,0.0,0,HB,PRT,...,13.37,5.81,1746.0,167738.0,42.49,213641489.9,412368.0,62.74,67.73,58.9


####10.4 —  Validación final de la unión

In [44]:
print("Dimensiones del dataset:", df_hb_to.shape)

print("\nRango de fechas:")
print("Min:", df_hb_to["date"].min())
print("Max:", df_hb_to["date"].max())

print("\nTipos de datos:")
print(df_hb_to.dtypes.head(10))

print("\nValores nulos en variables de indicadores:")
print(
    df_hb_to[
        [
            "empleos",
            "ingresos_totales",
            "tarifa_media_diaria",
            "tasa_de_ocupacion_por_habitacion"
        ]
    ].isnull().sum()
)

print("\nValores únicos de date (primeros 5):")
print(df_hb_to["date"].drop_duplicates().sort_values().head())

print("\nMuestra del dataset:")
df_hb_to.head()

Dimensiones del dataset: (116210, 50)

Rango de fechas:
Min: 2015-07-01 00:00:00
Max: 2017-08-01 00:00:00

Tipos de datos:
hotel                       object
is_canceled                  int64
lead_time                    int64
stays_in_weekend_nights      int64
stays_in_week_nights         int64
adults                       int64
children                   float64
babies                       int64
meal                        object
country                     object
dtype: object

Valores nulos en variables de indicadores:
empleos                             0
ingresos_totales                    0
tarifa_media_diaria                 0
tasa_de_ocupacion_por_habitacion    0
dtype: int64

Valores únicos de date (primeros 5):
0       2015-07-01
2712    2015-08-01
6479    2015-09-01
11488   2015-10-01
16310   2015-11-01
Name: date, dtype: datetime64[ns]

Muestra del dataset:


,hotel,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,...,empleos_por_cada_100_plazas_alojativas,empleos_por_cada_100_viajeros_entrados,establecimientos,habitaciones,ingresos_por_habitacion,ingresos_totales,plazas,tarifa_media_diaria,tasa_de_ocupacion_por_habitacion,tasa_de_ocupacion_por_plaza
0,Resort Hotel,0,7,0,1,1,0.0,0,BB,GBR,...,13.37,5.81,1746.0,167738.0,42.49,213641489.9,412368.0,62.74,67.73,58.9
1,City Hotel,0,257,0,2,1,0.0,0,HB,PRT,...,13.37,5.81,1746.0,167738.0,42.49,213641489.9,412368.0,62.74,67.73,58.9
2,City Hotel,0,257,0,2,2,0.0,0,HB,PRT,...,13.37,5.81,1746.0,167738.0,42.49,213641489.9,412368.0,62.74,67.73,58.9
3,City Hotel,0,257,0,2,2,0.0,0,HB,PRT,...,13.37,5.81,1746.0,167738.0,42.49,213641489.9,412368.0,62.74,67.73,58.9
4,City Hotel,0,257,0,2,2,0.0,0,HB,PRT,...,13.37,5.81,1746.0,167738.0,42.49,213641489.9,412368.0,62.74,67.73,58.9


### 11 — Exportado de datos

In [45]:
df_ta_wide.to_csv("tourist_occupation_prepared.csv", index=False)

In [46]:
df_hb_to.to_csv("bookings_tourists.csv", index=False)